## ⚙️ Environment Setup (run once)

A maioria dos serviços AgentCore (Gateway, Memory, Runtime, Registry) requer
versões recentes de `boto3`. Esta cislula instala/atualiza tudo o que o
workshop precisa.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 02.1 — Create Gateway with JWT Authorizer

## Overview

We will create um **AgentCore Gateway** que aceita JWTs do Cognito (Lab 01)
e expõe MCP tools. O Gateway is o ponto de entrada onde:

- JWT Tokens are validated before any call
- Cedar policies (Lab 03) decidem se a chamada is PERMIT ou DENY
- Lambdas (next notebook) ficam por trás como targets MCP

> 🎯 **Aqui is onde AgentCore Identity entra em cena de verdade.**  
> O `customJWTAuthorizer` that we will configure below is literally
> AgentCore Identity *inbound auth*. Ele consome o IdP externo (Cognito,
> que criamos no Lab 01) e cria o **Workload Identity** do Gateway —
> all of this happens behind the scenes when we pass
> `authorizerType=CUSTOM_JWT`.

## Tutorial Details

| Information | Details |
|---|---|
| Tutorial type | Interactive |
| AgentCore components | Gateway + Identity (do Lab 01) |
| Complexity | Medium |
| SDK | boto3 |
| Estimated time | 8 minutes |

## Prerequisites

- ✅ [Lab 01 — AgentCore Identity](../01-Identity-Foundation/) concluído
- `config.env` deve ter `COGNITO_USER_POOL_ID`, `COGNITO_CLIENT_ID`, `COGNITO_DISCOVERY_URL`

## Setup

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from shared.utils.config import load_config, save_config, get_region
from shared.utils.iam import create_gateway_role
from utils import create_gateway_with_jwt_authorizer, wait_for_gateway_ready

cfg = load_config()
region = get_region()
print(f"Pool ID: {cfg.get('COGNITO_USER_POOL_ID')}")
print(f"Client ID: {cfg.get('COGNITO_CLIENT_ID')}")
print(f"Discovery URL: {cfg.get('COGNITO_DISCOVERY_URL')}")

## Step 1: Criar IAM role para o Gateway

The Gateway needs a role assumed by the service `bedrock-agentcore.amazonaws.com`
with permission to invoke Lambdas and access agent-credential-provider.

In [ ]:
role_arn = create_gateway_role("workshop-gateway-role")
print(f"\nGateway role ARN: {role_arn}")

## Step 2: Criar o Gateway com JWT authorizer

Configuração-chave (`authorizerConfiguration`):
- **`discoveryUrl`** — endpoint OIDC do Cognito (do Lab 01)
- **`allowedScopes`** — only accepts JWTs with at least one of these scopes
>
> ℹ️ Neste workshop o login usa `USER_PASSWORD_AUTH`, cujo token carrega
> apenas o scope `aws.cognito.signin.user.admin` (ver nota no Lab 01.2).
> Por isso o Gateway is configurado para aceitar exatamente esse scope;
> fine-grained per-persona authorization is done by Cedar (Lab 03), not by scopes.
- **`customClaims`** — additional validation: `token_use=access` (rejects id tokens)
- **`allowedClients`** — restringe a app clients específicos

> 💡 **AgentCore Identity in action.** When you create the Gateway with
> `authorizerType=CUSTOM_JWT`, o serviço internamente:
> 1. Cria uma **Workload Identity** para o Gateway no Workload Identity
>    Directory `default` da sua conta
> 2. Configura o IdP externo (Cognito) como source de tokens válidos
> 3. Aplica os scopes/claims como filtros em cada chamada
>  
> This is the same mechanism used by the Runtime in Lab 05 — both consume
> AgentCore Identity *inbound auth* via essa mesma config.

In [ ]:
result = create_gateway_with_jwt_authorizer(
    name="workshop-gateway",
    role_arn=role_arn,
    discovery_url=cfg["COGNITO_DISCOVERY_URL"],
    allowed_clients=[cfg["COGNITO_CLIENT_ID"]],
    region=region,
)
print(json.dumps(result, indent=2))

## Step 3: Aguardar o Gateway atingir status READY

A criação is assíncrona — o Gateway demora ~30-60 segundos para provisionar
infra subjacente.

In [ ]:
status = wait_for_gateway_ready(result["gateway_id"], region=region)
print(f"\n✓ Gateway pronto: {status}")

## Step 4: Persist IDs to config.env

In [ ]:
save_config({
    "GATEWAY_ID": result["gateway_id"],
    "GATEWAY_ARN": result["gateway_arn"],
    "GATEWAY_URL": result["gateway_url"],
    "GATEWAY_ROLE_ARN": role_arn,
})

## ✅ Validation

O Gateway agora rejeita qualquer chamada sem um JWT válido. We will confirmar
adicionando Lambdas e tentando chamar (com e sem token) no next notebook.

In [ ]:
# Verificação simples — gateway URL deve ter formato MCP
mcp_url = result["gateway_url"]
if mcp_url and "mcp" in mcp_url.lower():
    print(f"✓ Gateway URL OK: {mcp_url}")
else:
    print(f"⚠ URL inesperada: {mcp_url}")

## 🎓 What you learned

- Gateway com `CUSTOM_JWT` integra-se com qualquer IdP que exponha OIDC discovery
- `customClaims.token_use=access` previne uso indevido de id tokens
- `allowedScopes` restringe quais clients podem chamar tools

## Next

➡️ [02.2 — Add Lambda Targets](./02-add-lambda-targets.ipynb)